# ACCT 3343 — Chapter 10: Backtesting accounting strategies

**Instructor notebook — simplified for a first coding class**

Today we will turn accounting ratios into investment rules using our **same 50 companies**. The next-year stock returns are already in the CSV, so we can concentrate on the accounting decisions rather than downloading and cleaning share prices.

**Four exercises:** (1) buy high-GPA companies, (2) buy companies with high GPA and low accruals, (3) short companies with high accruals, and (4) buy high-GPA and short low-GPA companies.

Run each cell from top to bottom in Google Colab. No installation or authentication is needed for the public CSV.

## 1. Load **one CSV**

Each row is a company at a July portfolio-formation date. The accounting values are selected using a **six-month reporting lag**; `stock_return` measures the *subsequent* July-to-June adjusted-price holding-period return.

This is a classroom illustration rather than a truly point-in-time database: historical statements may have been revised, and the 50-company universe was selected using today's surviving companies. We won't pretend it is a definitive investment test.

In [ ]:
import math
import pandas as pd

url = "https://raw.githubusercontent.com/x2003xuy/acct3343-public/main/data/acct3343_backtest_panel_50.csv"
df = pd.read_csv(url)

# The 2022 rows lack useful GPA and STA observations.
df = df[df["formation_year"] >= 2023].copy()

display(df[["ticker", "formation_year", "gross_profit", "total_assets",
            "net_income", "operating_cash_flow", "stock_return"]].head(10))

## 2. Calculate two accounting ratios

**Gross profitability (GPA)** = gross profit ÷ total assets. **Higher** GPA is generally considered attractive.

**Scaled total accruals (STA)** = (net income − operating cash flow) ÷ total assets. **Lower** STA indicates stronger cash backing of earnings under this simple proxy.

Only calculate a ratio when the necessary figures are available and assets are positive. `NaN` means *missing*, not zero.

In [ ]:
assets = df["total_assets"].where(df["total_assets"] > 0)
df["GPA"] = df["gross_profit"] / assets
df["STA"] = (df["net_income"] - df["operating_cash_flow"]) / assets

display(df[["ticker", "formation_year", "GPA", "STA", "stock_return"]].head(12))

## 3. Strategy 1 — Buy the top 10% by GPA

For **each July formation year separately**, sort companies from highest to lowest GPA. Buy the top `ceil(10% × eligible firms)` with **equal dollar weights**.

A long-only portfolio's one-year return is simply the **average** of its stocks' subsequent returns.

In [ ]:
results1 = []
holdings1 = []

for year, group in df.groupby("formation_year"):
    firms = group.dropna(subset=["GPA", "stock_return"]).copy()
    firms = firms.sort_values(["GPA", "ticker"], ascending=[False, True])
    n = math.ceil(len(firms) * 0.10)
    picks = firms.head(n)

    print(f"\n{year}: bought {n} of {len(firms)} GPA-eligible firms")
    display(picks[["ticker", "GPA", "stock_return"]])

    results1.append({"year": year, "GPA long": picks["stock_return"].mean()})
    holdings1.append(picks)

strategy1 = pd.DataFrame(results1).set_index("year")
display(strategy1.style.format("{:.2%}"))

## 4. Strategy 2 — Combine GPA and STA

Prefer both **high GPA** and **low STA**. To put the ratios on a common scale, rank GPA from high to low and STA from low to high. **Average the two ranks**; a smaller average rank is better.

Importantly, only companies with **both** valid ratios compete. Ranking by each ratio separately and then intersecting the top deciles is a *different* rule.

In [ ]:
results2 = []

for year, group in df.groupby("formation_year"):
    firms = group.dropna(subset=["GPA", "STA", "stock_return"]).copy()

    firms["GPA_rank"] = firms["GPA"].rank(ascending=False, method="min")
    firms["STA_rank"] = firms["STA"].rank(ascending=True, method="min")
    firms["combined_rank"] = (firms["GPA_rank"] + firms["STA_rank"]) / 2

    firms = firms.sort_values(["combined_rank", "ticker"])
    n = math.ceil(len(firms) * 0.10)
    picks = firms.head(n)

    print(f"\n{year}: bought {n} of {len(firms)} eligible firms")
    display(picks[["ticker", "GPA", "STA", "combined_rank", "stock_return"]])

    results2.append({"year": year, "GPA + STA long": picks["stock_return"].mean()})

strategy2 = pd.DataFrame(results2).set_index("year")
display(strategy2.style.format("{:.2%}"))

## 5. Strategy 3 — Short the top 10% by STA

Select companies with the **highest** STA (poorest accrual quality under this proxy). This strategy uses **STA alone**—not GPA.

If an underlying stock rises 20%, a short position loses approximately 20% of its starting notional. If it falls 20%, the short gains approximately 20%. Thus the teaching-model short return is **minus the stock's adjusted total return**. We assume total short notional equals initial investor equity, with no costs or margin constraints.

In [ ]:
results3 = []

for year, group in df.groupby("formation_year"):
    firms = group.dropna(subset=["STA", "stock_return"]).copy()
    firms = firms.sort_values(["STA", "ticker"], ascending=[False, True])
    n = math.ceil(len(firms) * 0.10)
    picks = firms.head(n).copy()
    picks["short_return"] = -picks["stock_return"]

    print(f"\n{year}: shorted {n} of {len(firms)} STA-eligible firms")
    display(picks[["ticker", "STA", "stock_return", "short_return"]])

    results3.append({"year": year, "STA short": picks["short_return"].mean()})

strategy3 = pd.DataFrame(results3).set_index("year")
display(strategy3.style.format("{:.2%}"))

## 6. Strategy 4 — Buy high GPA, short low GPA

Using the **same GPA-eligible firms** in each year: buy the top decile and short the bottom decile.

Allocate **50% of starting equity to the long side** and **50% as short notional**. Therefore the portfolio return is:

**0.5 × average high-GPA stock return − 0.5 × average low-GPA stock return**

Don't use 100% on *each* side—that would double the gross exposure.

In [ ]:
results4 = []

for year, group in df.groupby("formation_year"):
    firms = group.dropna(subset=["GPA", "stock_return"]).copy()
    high_to_low = firms.sort_values(["GPA", "ticker"], ascending=[False, True])
    low_to_high = firms.sort_values(["GPA", "ticker"], ascending=[True, True])
    n = math.ceil(len(firms) * 0.10)

    long_picks = high_to_low.head(n)
    short_picks = low_to_high.head(n)
    high_return = long_picks["stock_return"].mean()
    low_return = short_picks["stock_return"].mean()
    portfolio_return = 0.5 * high_return - 0.5 * low_return

    print(f"\n{year}: {n} longs and {n} shorts")
    print("Longs:", ", ".join(long_picks["ticker"]))
    print("Shorts:", ", ".join(short_picks["ticker"]))

    results4.append({"year": year, "GPA long-short": portfolio_return})

strategy4 = pd.DataFrame(results4).set_index("year")
display(strategy4.style.format("{:.2%}"))

## 7. Compare the four strategies

The annual strategy returns are **one-year holding-period returns**, not a prediction or evidence of causality.

For cumulative growth, imagine investing **$1 at the start of 2023**, reinvesting each July, and compounding by `1 + annual_return`. In this observed sample, all four series keep positive modeled equity. Real short selling has additional margin and borrowing risks that this simple illustration ignores.

In [ ]:
results = pd.concat([strategy1, strategy2, strategy3, strategy4], axis=1)
display(results.style.format("{:.2%}"))

# Annual returns (percent)
ax = (results * 100).plot.bar(figsize=(11, 5))
ax.set_title("Return earned during each July-to-June period")
ax.set_xlabel("July formation year")
ax.set_ylabel("Portfolio return (%)")
ax.axhline(0, color="black", linewidth=0.8)
ax.legend(loc="best")
display(ax.figure)
import matplotlib.pyplot as plt
plt.close(ax.figure)

# Cumulative value of $1 invested in each strategy
wealth = (1 + results).cumprod()
wealth.loc[2022] = 1.0
wealth = wealth.sort_index()

ax = wealth.plot(figsize=(11, 5), marker="o")
ax.set_title("Illustrative growth of $1 (annual compounding)")
ax.set_xlabel("July formation year (2022 = starting $1)")
ax.set_ylabel("Portfolio value ($)")
ax.grid(alpha=0.25)
display(ax.figure)
plt.close(ax.figure)

## 8. Discussion: what would make this a stronger backtest?

- **Data timing:** These rows use a six-month lag, not verified first-release values for every company. Restated figures may differ from what investors saw at the time.
- **Survivorship bias:** These 50 present-day companies aren't the complete investable universe in 2023–2025.
- **Very small sample:** We have just three one-year investment periods; some legs hold only four or five companies.
- **Different businesses:** The meaning of GPA and STA can differ for financial and nonfinancial companies.
- **Real-world trading:** Borrowing costs, short-sale availability, margins, taxes, transaction costs, and liquidity are not modeled.
- **Benchmarks:** The class competition compared a portfolio against the S&P 500. Here we compare *four strategies on the same 50-company universe*; this file doesn't contain an S&P 500 return series.

**Questions for students:** Which strategy would you have chosen before seeing the returns? Did the returns support it? What would you need to observe before claiming the rule genuinely works?

For the full research-style implementation, see `ACCT3343_Chapter10_Backtesting_Advanced.ipynb` in this private repository.